# Month-end finance analysis
Revenue, DSO trend, collections vs billing, and overdue accounts. Run from this directory. Amounts assume a single reporting currency; dates are evaluated in UTC on Snowflake (SFX_D1110_AFTER.COMPAT). Rolling windows include the current partial month.

In [ ]:
from finance_nb.warehouse import connect, run_query

# Snowflake key-pair settings come from SNOWFLAKE_ACCOUNT, SNOWFLAKE_USER,
# SNOWFLAKE_ROLE, SNOWFLAKE_WAREHOUSE and SNOWFLAKE_PRIVATE_KEY.
connection = connect()

## Revenue by month

In [ ]:
revenue_sql = """SELECT TO_CHAR(INVOICE_DT, 'YYYY-MM') AS MONTH_LABEL,
  SUM(TOTAL_AMOUNT - COALESCE(TAX_AMOUNT, 0)) AS REVENUE
FROM SFX_D1110_AFTER.COMPAT.INVOICE_HDR
WHERE INVOICE_DT >= ADD_MONTHS(CURRENT_DATE(), -12) AND INVOICE_DT >= DATE '2020-01-01'
GROUP BY 1 ORDER BY 1"""
revenue = run_query(connection, revenue_sql)
revenue

## DSO trend

In [ ]:
dso_sql = """WITH billed AS (
  SELECT TO_CHAR(INVOICE_DT, 'YYYY-MM') AS MONTH_LABEL,
    LAST_DAY(DATE_TRUNC('MONTH', INVOICE_DT)) AS MONTH_END,
    SUM(TOTAL_AMOUNT) AS BILLED
  FROM SFX_D1110_AFTER.COMPAT.INVOICE_HDR
  WHERE INVOICE_DT >= ADD_MONTHS(CURRENT_DATE(), -12) AND INVOICE_DT >= DATE '2020-01-01'
  GROUP BY 1, 2
), paid AS (
  SELECT b.MONTH_LABEL, p.INVOICE_ID, SUM(p.AMOUNT) AS PAID
  FROM billed b JOIN SFX_D1110_AFTER.COMPAT.PAYMENT p ON p.PAYMENT_DT <= b.MONTH_END
  GROUP BY 1, 2
), open_ar AS (
  SELECT b.MONTH_LABEL, SUM(i.TOTAL_AMOUNT - COALESCE(p.PAID, 0)) AS OPEN_AR
  FROM billed b
  JOIN SFX_D1110_AFTER.COMPAT.INVOICE_HDR i ON i.INVOICE_DT <= b.MONTH_END
  LEFT JOIN paid p ON p.INVOICE_ID = i.INVOICE_ID AND p.MONTH_LABEL = b.MONTH_LABEL
  WHERE i.INVOICE_DT >= DATE '2020-01-01'
  GROUP BY 1
)
SELECT b.MONTH_LABEL, COALESCE(a.OPEN_AR / NULLIF(b.BILLED, 0) * 30, 0) AS DSO
FROM billed b JOIN open_ar a ON a.MONTH_LABEL = b.MONTH_LABEL
ORDER BY 1"""
dso = run_query(connection, dso_sql)
dso

## Collections vs billing

In [ ]:
collections_sql = """WITH billed AS (
  SELECT TO_CHAR(INVOICE_DT, 'YYYY-MM') AS MONTH_LABEL, SUM(TOTAL_AMOUNT) AS BILLED
  FROM SFX_D1110_AFTER.COMPAT.INVOICE_HDR
  WHERE INVOICE_DT >= DATEADD(DAY, -90, CURRENT_DATE()) AND INVOICE_DT >= DATE '2020-01-01'
  GROUP BY 1
), cash AS (
  SELECT TO_CHAR(PAYMENT_DT, 'YYYY-MM') AS MONTH_LABEL, SUM(AMOUNT) AS COLLECTED
  FROM SFX_D1110_AFTER.COMPAT.PAYMENT
  WHERE PAYMENT_DT >= DATEADD(DAY, -90, CURRENT_DATE())
  GROUP BY 1
)
SELECT COALESCE(b.MONTH_LABEL, c.MONTH_LABEL) AS MONTH_LABEL,
  COALESCE(b.BILLED, 0) AS BILLED, COALESCE(c.COLLECTED, 0) AS COLLECTED
FROM billed b FULL OUTER JOIN cash c ON c.MONTH_LABEL = b.MONTH_LABEL
ORDER BY 1"""
collections = run_query(connection, collections_sql)
collections

## Top overdue accounts

In [ ]:
overdue_sql = """WITH paid AS (
  SELECT INVOICE_ID, SUM(AMOUNT) AS PAID
  FROM SFX_D1110_AFTER.COMPAT.PAYMENT GROUP BY INVOICE_ID
), balances AS (
  SELECT i.CUSTOMER_ID, DATEADD(DAY, 30, i.INVOICE_DT) AS DUE_DT,
    i.TOTAL_AMOUNT - COALESCE(p.PAID, 0) AS OPEN_BALANCE
  FROM SFX_D1110_AFTER.COMPAT.INVOICE_HDR i
  LEFT JOIN paid p ON p.INVOICE_ID = i.INVOICE_ID
  WHERE i.INVOICE_DT >= DATE '2020-01-01'
    AND DATEADD(DAY, 30, i.INVOICE_DT) < DATEADD(DAY, -30, CURRENT_DATE())
)
SELECT a.ACCOUNT_NUMBER AS ACCOUNT_NO, a.ACCOUNT_NAME AS CUSTOMER_NAME,
  MIN(b.DUE_DT) AS OLDEST_DUE_DATE, SUM(b.OPEN_BALANCE) AS OVERDUE_BALANCE
FROM balances b
JOIN SFX_D1110_AFTER.COMPAT.CUSTOMER_ACCOUNT a ON a.CUSTOMER_ID = b.CUSTOMER_ID
WHERE b.OPEN_BALANCE > 0
GROUP BY UPPER(a.ACCOUNT_NUMBER), a.ACCOUNT_NUMBER, a.ACCOUNT_NAME
ORDER BY OVERDUE_BALANCE DESC, UPPER(a.ACCOUNT_NUMBER), a.ACCOUNT_NUMBER
LIMIT 10"""
overdue = run_query(connection, overdue_sql)
overdue

In [ ]:
connection.close()